In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install earthengine-api geemap rasterio scikit-image

In [ ]:
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
ee.Authenticate()
ee.Initialize(project='YOUR_EE_PROJECT_ID')

In [ ]:
california_roi = ee.Geometry.Rectangle([

    -124.5,
    36.5,

    -119.0,
    41.5
])

In [ ]:
Map = geemap.Map()

Map.centerObject(california_roi, 6)

Map.addLayer(california_roi, {}, "California ROI")

Map

In [ ]:
ndvi_collection = ee.ImageCollection("MODIS/061/MOD13Q1")

In [ ]:
ndvi = (
    ndvi_collection
    .filterBounds(california_roi)
    .filterDate("2025-06-01", "2025-06-30")
    .select("NDVI")
    .mean()
)

In [ ]:
print(ndvi.getInfo())

In [ ]:
ndvi = ndvi.multiply(0.0001)

In [ ]:
Map = geemap.Map()

Map.centerObject(california_roi, 6)

ndvi_vis = {
    "min": 0,
    "max": 1,
    "palette": ["white", "green"]
}

Map.addLayer(ndvi.clip(california_roi), ndvi_vis, "California NDVI")

Map

In [ ]:
export_path = "/content/california_ndvi.tif"

geemap.ee_export_image(
    ndvi.clip(california_roi),
    filename=export_path,
    scale=500,
    region=california_roi,
    file_per_band=False
)

print("California NDVI exported successfully.")

In [ ]:
import rasterio

src = rasterio.open(export_path)

ndvi_array = src.read(1)

print(ndvi_array.shape)

In [ ]:
plt.figure(figsize=(8,6))

plt.imshow(ndvi_array, cmap="Greens")

plt.title("Raw California NDVI")

plt.colorbar()

plt.show()

In [ ]:
ndvi_array = np.nan_to_num(ndvi_array)

ndvi_array[ndvi_array < 0] = 0

In [ ]:
def normalize(x):
    return (x - x.min()) / (x.max() - x.min())

In [ ]:
fuel = normalize(ndvi_array)

In [ ]:
print(fuel.min())
print(fuel.max())

In [ ]:
plt.figure(figsize=(8,6))

plt.imshow(fuel, cmap="Greens")

plt.title("Normalized California Fuel Layer")

plt.colorbar()

plt.show()

In [ ]:
from skimage.transform import resize

In [ ]:
GRID_32 = (32,32)

GRID_64 = (64,64)

In [ ]:
fuel_32 = resize(fuel, GRID_32)

fuel_64 = resize(fuel, GRID_64)

In [ ]:
print(fuel_32.shape)

print(fuel_64.shape)

In [ ]:
plt.figure(figsize=(6,6))

plt.imshow(fuel_32, cmap="Greens")

plt.title("California Fuel Grid 32x32")

plt.colorbar()

plt.show()

In [ ]:
BASE = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/california/grids"

os.makedirs(f"{BASE}/32x32", exist_ok=True)

os.makedirs(f"{BASE}/64x64", exist_ok=True)

In [ ]:
np.save(f"{BASE}/32x32/fuel.npy", fuel_32)

print("California fuel 32x32 saved.")

In [ ]:
np.save(f"{BASE}/64x64/fuel.npy", fuel_64)

print("California fuel 64x64 saved.")

In [ ]:
grid_path = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/california/grids/32x32"

print(os.listdir(grid_path))

In [ ]:
fuel_loaded = np.load(f"{grid_path}/fuel.npy")

print(fuel_loaded.shape)

print(fuel_loaded.min())

print(fuel_loaded.max())